# RotorQuant Local Quantisation
**No Colab needed.** This notebook handles two local workflows:
1. **Path A — LoRA Conversion**: Converts raw `safetensors` LoRA adapters (from training) to GGUF. (⚠️ Not yet supported for Gemma 4 VLM architecture).
2. **Path B — Quantisation**: Re-quantises the existing merged `gemma4-legal-vlm` Ollama blob
(which already has the legal fine-tune baked in) from Q4_K_M → IQ4_XS so it can run under the
RotorQuant / AtomicBot inference stack.

### What this does
1. Locates the Ollama blob on disk (it is already a valid GGUF file)
2. Copies it with a readable name
3. Calls `llama-quantize.exe` with `--allow-requantize` to convert to IQ4_XS
4. Prints the env var line to paste into `.env` 

### Prerequisites
- `llama-quantize.exe` from the same build as your `llama-server.exe`
  (both live in `C:\Users\james\Desktop\llama-server-cuda\` by default)
- ~10 GB free disk space

In [12]:
import os, shutil, subprocess, pathlib, getpass

# ── Where llama-quantize.exe lives (same folder as llama-server.exe) ──────────
LLAMA_QUANTIZE = os.environ.get(
    "LLAMA_QUANTIZE_PATH",
    r"C:\Users\james\Desktop\llama-server-cuda\llama-quantize.exe"
)

# ── Source GGUF — we already found these on disk, pick whichever exists ────────
# Priority: Downloads folder first (more likely to be the freshest copy)
_CANDIDATES = [
    r"C:\Users\james\Downloads\gemma4-legal-vlm-q4_k_m.gguf",
    r"C:\Users\james\Downloads\gemma4-e4b-legal-ollama\gemma4-e4b-legal.Q4_K_M.gguf",
    # Ollama blob fallback (works but needs a copy step first)
    os.path.expandvars(
        r"%USERPROFILE%\.ollama\models\blobs"
        r"\sha256-a79de882a921b9c3781a95a8ef555ea51e7c4dd685a8b2854e9bbe73ab081b43"
    ),
]
SOURCE_GGUF = next((p for p in _CANDIDATES if os.path.exists(p)), None)

# ── Output directory ───────────────────────────────────────────────────────────
OUT_DIR = pathlib.Path(os.path.expandvars(r"%USERPROFILE%\Desktop\gemma4-legal-iq4xs"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

Q4KM_PATH  = OUT_DIR / "gemma4-legal-merged-q4km.gguf"
IQ4XS_PATH = OUT_DIR / "gemma4-legal-iq4xs.gguf"

# ── Checks ─────────────────────────────────────────────────────────────────────
assert SOURCE_GGUF is not None, (
    "No source GGUF found in the candidate paths above.\n"
    "Add the full path to your merged gemma4-legal GGUF to _CANDIDATES."
)
assert os.path.exists(LLAMA_QUANTIZE), (
    f"llama-quantize.exe not found at:\n  {LLAMA_QUANTIZE}\n"
    "Set the LLAMA_QUANTIZE_PATH env var or edit the path above."
)

src_gb = os.path.getsize(SOURCE_GGUF) / 1e9
print(f"✅ Source GGUF:         {SOURCE_GGUF}  ({src_gb:.1f} GB)")
print(f"✅ llama-quantize:      {LLAMA_QUANTIZE}")
print(f"📁 Output directory:   {OUT_DIR}")
print()
print("ℹ️  All files found locally — no Colab required.")

✅ Source GGUF:         C:\Users\james\Downloads\gemma4-legal-vlm-q4_k_m.gguf  (5.3 GB)
✅ llama-quantize:      C:\Users\james\Desktop\llama-server-cuda\llama-quantize.exe
📁 Output directory:   C:\Users\james\Desktop\gemma4-legal-iq4xs

ℹ️  All files found locally — no Colab required.


## Step 1 — Copy source model
We copy the model to a workspace folder with a readable name.

In [13]:
if Q4KM_PATH.exists():
    print(f"⏭  Already copied: {Q4KM_PATH} ({Q4KM_PATH.stat().st_size / 1e9:.1f} GB)")
else:
    print(f"Copying source → {Q4KM_PATH}  (may take a minute)...")
    shutil.copy2(SOURCE_GGUF, Q4KM_PATH)
    print(f"✅ Copied: {Q4KM_PATH.stat().st_size / 1e9:.1f} GB")


⏭  Already copied: C:\Users\james\Desktop\gemma4-legal-iq4xs\gemma4-legal-merged-q4km.gguf (5.3 GB)


## Step 2 — Quantise to IQ4_XS
IQ4_XS uses importance-weighted block-diagonal quantisation — smaller than Q4_K_M with
comparable quality. 

⚠️ We use **`--allow-requantize`** because the source model already has some quantized tensors (like token embeddings).
Takes ~3-5 min. Output is ~4.5 GB.

In [14]:
if IQ4XS_PATH.exists() and IQ4XS_PATH.stat().st_size < 4 * 1024**3:
    print(f"⚠️ Removing incomplete IQ4_XS output: {IQ4XS_PATH} ({IQ4XS_PATH.stat().st_size / 1e9:.1f} GB)")
    IQ4XS_PATH.unlink()
if IQ4XS_PATH.exists():
    print(f"⏭  Already quantised: {IQ4XS_PATH} ({IQ4XS_PATH.stat().st_size / 1e9:.1f} GB)")
else:
    print(f"Quantising to IQ4_XS (direct re-quantisation)...")
    result = subprocess.run(
        [LLAMA_QUANTIZE, "--allow-requantize", str(Q4KM_PATH), str(IQ4XS_PATH), "IQ4_XS"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"llama-quantize failed (exit {result.returncode})")
    print(f"\n✅ IQ4_XS written: {IQ4XS_PATH} ({IQ4XS_PATH.stat().st_size / 1e9:.1f} GB)")


Quantising to IQ4_XS (direct re-quantisation)...

✅ IQ4_XS written: C:\Users\james\Desktop\gemma4-legal-iq4xs\gemma4-legal-iq4xs.gguf (5.1 GB)


## Step 3 — Output .env line
Copy the line below into `sveltekit-frontend/.env` then run `npm run turbo:start:rotorquant`.

In [15]:
if IQ4XS_PATH.exists():
    env_line = f"ROTORQUANT_MODEL_PATH={IQ4XS_PATH}"
    print("=" * 60)
    print("Paste this into sveltekit-frontend/.env:")
    print()
    print(env_line)
    print()
    print("=" * 60)
    print()
    print("Then run:")
    print("  npm run turbo:start:rotorquant")
    print("  npm run turbo:bench:rotorquant")
    print()
else:
    print("❌ IQ4_XS file not found — run Step 2 first.")


Paste this into sveltekit-frontend/.env:

ROTORQUANT_MODEL_PATH=C:\Users\james\Desktop\gemma4-legal-iq4xs\gemma4-legal-iq4xs.gguf


Then run:
  npm run turbo:start:rotorquant
  npm run turbo:bench:rotorquant

